# Module 10 • Advanced Applications
# Lesson 57 • Advanced Information Retrieval — Neural Retrieval, Re-Ranking, and Production Search Pipelines

**Course:** Natural Language Processing: From Foundations to Large Language Models  
**Author:** Eman Khater  
**Execution target:** CPU only

## Scope
Build a modern multi-stage search system with lexical retrieval, dense-like semantic
retrieval, hybrid fusion, re-ranking, ranking metrics, latency measurement, Arabic
considerations, and production architecture.

The notebook runs fully offline. Truncated SVD is used as a transparent dense-like
stand-in for neural embeddings; real bi-encoder/cross-encoder integration is shown as
an optional disabled template.

## Learning Objectives
- distinguish lexical, dense, hybrid, and reranked retrieval;
- implement BM25-style ranking;
- build a latent-semantic dense retriever;
- fuse rankings;
- train a simple reranker;
- evaluate Precision@k, Recall@k, MRR, MAP, and nDCG;
- analyze latency and failure modes;
- design multilingual/Arabic search;
- map a notebook prototype to a production search pipeline.

## Table of Contents
1. Modern IR Architecture
2. Lexical Retrieval
3. BM25
4. Dense Retrieval
5. Bi-Encoders and Cross-Encoders
6. Hybrid Retrieval
7. Re-Ranking
8. Search Corpus
9. Relevance Judgments
10. TF-IDF Index
11. BM25 Index
12. Latent Semantic Index
13. Sparse Search
14. Dense-Like Search
15. Hybrid Fusion
16. Reciprocal Rank Fusion
17. Metadata Filtering
18. Re-Ranker Features
19. Learning-to-Rank Re-Ranker
20. End-to-End Search
21. Precision@k and Recall@k
22. MRR
23. MAP
24. nDCG
25. Method Comparison
26. Re-Ranking Evaluation
27. Query-Level Analysis
28. Failure Analysis
29. Multilingual Retrieval
30. Arabic Retrieval
31. Tashkeel Policy
32. Latency
33. Caching and Index Freshness
34. Production Architecture
35. Monitoring and Access Control
36. Optional Neural Retrieval
37. Reproducibility
38. Knowledge Check
39. Exercises
40. Summary and Next Lesson

# 1. Modern IR Architecture

Modern search is commonly multi-stage:

```text
query
  ↓
normalization
  ↓
candidate generation
  ├── lexical retriever
  └── dense retriever
  ↓
fusion
  ↓
top-N candidates
  ↓
reranker
  ↓
final top-k
```

Candidate generation emphasizes recall and speed; re-ranking emphasizes precision.

In [ ]:
import math, platform, re, time
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer, StandardScaler

SEED = 42
np.random.seed(SEED)

pd.DataFrame([
    ("Lexical", "exact token evidence", "BM25 / TF-IDF"),
    ("Dense", "semantic vector similarity", "bi-encoder"),
    ("Hybrid", "combine complementary signals", "fusion / RRF"),
    ("Re-ranking", "deeper top-N scoring", "cross-encoder / LTR"),
], columns=["Stage", "Purpose", "Typical method"])

# 2. Lexical Retrieval
Lexical methods are strong when queries and relevant documents share informative terms.
They can struggle with synonyms and paraphrases.

# 3. BM25
BM25 combines inverse document frequency, term-frequency saturation, and
document-length normalization.

```text
score(q,d) = Σ IDF(t) × tf(t,d)(k1+1) /
             [tf(t,d) + k1(1-b+b|d|/avgdl)]
```

# 4. Dense Retrieval
Dense retrieval embeds queries and documents into continuous vectors. Similarity can
recover semantic matches even when exact vocabulary differs.

# 5. Bi-Encoders and Cross-Encoders
A **bi-encoder** independently encodes query and documents, so document vectors can be
precomputed. A **cross-encoder** jointly scores a query-document pair and is usually
more expensive but stronger for re-ranking.

# 6. Hybrid Retrieval
Hybrid retrieval combines sparse and dense evidence because their failure modes differ.

# 7. Re-Ranking
A re-ranker receives a relatively small candidate set and produces a stronger final
ordering using richer features or a learned model.

# 8. Search Corpus

In [ ]:
documents = [
    ("d01","Transformer Architecture","transformers","en",
     "Transformers use self-attention to model token relationships and cross-attention in encoder-decoder models."),
    ("d02","Retrieval-Augmented Generation","rag","en",
     "Retrieval-augmented generation retrieves relevant evidence before generation and places passages in model context."),
    ("d03","BM25 Search","retrieval","en",
     "BM25 is a probabilistic lexical ranking function using inverse document frequency term saturation and document length normalization."),
    ("d04","Dense Passage Retrieval","retrieval","en",
     "Dense retrieval represents queries and passages as vectors so semantic similarity can match different wording."),
    ("d05","Cross-Encoder Re-Ranking","retrieval","en",
     "A cross-encoder jointly scores a query and candidate document and is commonly used to rerank a candidate set."),
    ("d06","LoRA Fine-Tuning","peft","en",
     "LoRA adapts a pretrained model with low-rank trainable matrices while keeping most weights frozen."),
    ("d07","Machine Translation Evaluation","mt","en",
     "Machine translation can be evaluated with BLEU chrF COMET confidence intervals and significance tests."),
    ("d08","Arabic Morphology","arabic","en",
     "Arabic is morphologically rich and contains attached clitics. Tokenization and diacritization affect NLP systems."),
    ("d09","Arabic Retrieval","arabic","ar",
     "يَعْتَمِدُ الِاسْتِرْجَاعُ النَّصِّيُّ عَلَى تَمْثِيلِ الِاسْتِعْلَامِ وَالْوَثَائِقِ وَحِسَابِ دَرَجَاتِ التَّشَابُهِ."),
    ("d10","Neural Search","retrieval","en",
     "Neural search learns semantic representations. Bi-encoders scale retrieval and cross-encoders improve reranking."),
    ("d11","Information Retrieval Metrics","evaluation","en",
     "Retrieval quality can be evaluated with precision recall reciprocal rank average precision MAP DCG and nDCG."),
    ("d12","Production Search Systems","systems","en",
     "Production search systems monitor latency index freshness relevance failures and access-control policies."),
]

docs = pd.DataFrame(documents, columns=["doc_id","title","topic","language","text"])
docs[["doc_id","title","topic","language"]]

# 9. Relevance Judgments

In [ ]:
queries = {
    "q01": "How does BM25 rank documents?",
    "q02": "semantic vector search for passages",
    "q03": "joint query document reranking",
    "q04": "metrics for evaluating search rankings",
    "q05": "production search latency and index freshness",
    "q06": "Arabic morphology and clitics",
}

relevance = {
    "q01": {"d03": 3, "d11": 1},
    "q02": {"d04": 3, "d10": 2},
    "q03": {"d05": 3, "d10": 2},
    "q04": {"d11": 3},
    "q05": {"d12": 3},
    "q06": {"d08": 3, "d09": 1},
}

pd.DataFrame([
    {"query_id": qid, "query": q, "relevant": list(relevance[qid])}
    for qid, q in queries.items()
])

# 10. TF-IDF Index

In [ ]:
TOKEN_PATTERN = re.compile(r"\b\w+\b", flags=re.UNICODE)

def normalize_text(text):
    return " ".join(TOKEN_PATTERN.findall(text.lower()))

vectorizer = TfidfVectorizer(
    preprocessor=normalize_text,
    ngram_range=(1, 2),
)
tfidf_matrix = vectorizer.fit_transform(docs["text"])
tfidf_matrix.shape

# 11. BM25 Index

In [ ]:
tokenized_docs = [normalize_text(text).split() for text in docs["text"]]
doc_lengths = np.array([len(x) for x in tokenized_docs], dtype=float)
avgdl = float(doc_lengths.mean())
df = Counter()
for tokens in tokenized_docs:
    for term in set(tokens):
        df[term] += 1

N = len(tokenized_docs)

def bm25_scores(query, k1=1.5, b=0.75):
    scores = np.zeros(N, dtype=float)
    qterms = normalize_text(query).split()

    for i, tokens in enumerate(tokenized_docs):
        counts = Counter(tokens)
        for term in qterms:
            tf = counts.get(term, 0)
            if tf == 0:
                continue
            idf = math.log(1 + (N - df.get(term, 0) + 0.5) / (df.get(term, 0) + 0.5))
            denom = tf + k1 * (1 - b + b * doc_lengths[i] / avgdl)
            scores[i] += idf * tf * (k1 + 1) / denom
    return scores

# 12. Latent Semantic Index

In [ ]:
n_components = min(8, tfidf_matrix.shape[0]-1, tfidf_matrix.shape[1]-1)
svd = TruncatedSVD(n_components=n_components, random_state=SEED)
latent_docs = svd.fit_transform(tfidf_matrix)
normalizer = Normalizer()
latent_docs = normalizer.fit_transform(latent_docs)

def tfidf_scores(query):
    q = vectorizer.transform([query])
    return cosine_similarity(q, tfidf_matrix)[0]

def latent_scores(query):
    q = vectorizer.transform([query])
    q_latent = normalizer.transform(svd.transform(q))
    return cosine_similarity(q_latent, latent_docs)[0]

# 13. Sparse Search

In [ ]:
def rank_scores(scores, top_k=5):
    idxs = np.argsort(scores)[::-1][:top_k]
    rows = []
    for rank, idx in enumerate(idxs, 1):
        row = docs.iloc[int(idx)]
        rows.append({
            "rank": rank,
            "score": float(scores[idx]),
            "doc_id": row.doc_id,
            "title": row.title,
            "topic": row.topic,
            "language": row.language,
            "text": row.text,
        })
    return pd.DataFrame(rows)

def bm25_search(query, top_k=5):
    return rank_scores(bm25_scores(query), top_k)

bm25_search(queries["q01"])

# 14. Dense-Like Search

In [ ]:
def latent_search(query, top_k=5):
    return rank_scores(latent_scores(query), top_k)

latent_search(queries["q02"])

# 15. Hybrid Fusion

In [ ]:
def minmax(values):
    values = np.asarray(values, dtype=float)
    lo, hi = values.min(), values.max()
    if hi - lo < 1e-12:
        return np.zeros_like(values)
    return (values - lo) / (hi - lo)

def hybrid_scores(query, lexical_weight=0.55):
    sparse = minmax(bm25_scores(query))
    dense = minmax(latent_scores(query))
    return lexical_weight * sparse + (1 - lexical_weight) * dense

def hybrid_search(query, top_k=5):
    return rank_scores(hybrid_scores(query), top_k)

hybrid_search(queries["q02"])

# 16. Reciprocal Rank Fusion

In [ ]:
def reciprocal_rank_fusion(rankings, rrf_k=60, top_k=5):
    fused = defaultdict(float)
    for ranking in rankings:
        for row in ranking.itertuples(index=False):
            fused[row.doc_id] += 1.0 / (rrf_k + row.rank)

    ordered = sorted(fused.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return pd.DataFrame([
        {
            "rank": rank,
            "doc_id": doc_id,
            "score": score,
            "title": docs.loc[docs.doc_id == doc_id, "title"].iloc[0],
        }
        for rank, (doc_id, score) in enumerate(ordered, 1)
    ])

reciprocal_rank_fusion([
    bm25_search(queries["q02"], 8),
    latent_search(queries["q02"], 8),
])

# 17. Metadata Filtering

In [ ]:
def filtered_hybrid_search(query, top_k=5, language=None, topic=None):
    scores = hybrid_scores(query).copy()
    mask = np.ones(len(docs), dtype=bool)
    if language is not None:
        mask &= docs["language"].to_numpy() == language
    if topic is not None:
        mask &= docs["topic"].to_numpy() == topic
    scores[~mask] = -np.inf
    return rank_scores(scores, top_k)

filtered_hybrid_search("Arabic morphology", top_k=2, topic="arabic")

# 18. Re-Ranker Features

In [ ]:
def token_overlap(query, text):
    q = set(normalize_text(query).split())
    d = set(normalize_text(text).split())
    return 0.0 if not q else len(q & d) / len(q)

def feature_vector(query, doc_index):
    return [
        bm25_scores(query)[doc_index],
        tfidf_scores(query)[doc_index],
        latent_scores(query)[doc_index],
        token_overlap(query, docs.iloc[doc_index]["text"]),
        math.log1p(len(tokenized_docs[doc_index])),
    ]

# 19. Learning-to-Rank Re-Ranker

In [ ]:
X, y = [], []
for qid, query in queries.items():
    rel = set(relevance[qid])
    for i in range(len(docs)):
        X.append(feature_vector(query, i))
        y.append(int(docs.iloc[i].doc_id in rel))

reranker = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        random_state=SEED,
        class_weight="balanced",
        max_iter=1000,
    ),
)
reranker.fit(np.asarray(X), np.asarray(y))

def rerank(query, candidates):
    features = []
    for row in candidates.itertuples(index=False):
        idx = int(docs.index[docs.doc_id == row.doc_id][0])
        features.append(feature_vector(query, idx))

    output = candidates.copy()
    output["rerank_score"] = reranker.predict_proba(np.asarray(features))[:, 1]
    output = output.sort_values("rerank_score", ascending=False).reset_index(drop=True)
    output["rank"] = np.arange(1, len(output)+1)
    return output

# 20. End-to-End Search

In [ ]:
def search_pipeline(query, candidate_k=8, final_k=5, rerank_results=True):
    candidates = hybrid_search(query, top_k=candidate_k)
    if rerank_results:
        candidates = rerank(query, candidates)
    return candidates.iloc[:final_k].reset_index(drop=True)

search_pipeline("joint query document scoring")

# 21. Precision@k and Recall@k

In [ ]:
def precision_at_k(ranking, relevant, k):
    return sum(x in relevant for x in ranking[:k]) / k

def recall_at_k(ranking, relevant, k):
    return 0.0 if not relevant else sum(x in relevant for x in ranking[:k]) / len(relevant)

# 22. MRR

In [ ]:
def reciprocal_rank(ranking, relevant):
    for rank, doc_id in enumerate(ranking, 1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0

# 23. MAP

In [ ]:
def average_precision(ranking, relevant):
    if not relevant:
        return 0.0
    hits, values = 0, []
    for rank, doc_id in enumerate(ranking, 1):
        if doc_id in relevant:
            hits += 1
            values.append(hits / rank)
    return sum(values) / len(relevant) if values else 0.0

# 24. nDCG

In [ ]:
def dcg_at_k(ranking, graded, k):
    total = 0.0
    for rank, doc_id in enumerate(ranking[:k], 1):
        grade = graded.get(doc_id, 0)
        total += (2**grade - 1) / math.log2(rank + 1)
    return total

def ndcg_at_k(ranking, graded, k):
    actual = dcg_at_k(ranking, graded, k)
    ideal = [d for d, _ in sorted(graded.items(), key=lambda x: x[1], reverse=True)]
    best = dcg_at_k(ideal, graded, k)
    return 0.0 if best == 0 else actual / best

# 25. Method Comparison

In [ ]:
def evaluate(search_fn):
    rows = []
    for qid, query in queries.items():
        result = search_fn(query, 5)
        ranking = result.doc_id.tolist()
        rel = set(relevance[qid])
        rows.append({
            "query_id": qid,
            "P@3": precision_at_k(ranking, rel, 3),
            "R@3": recall_at_k(ranking, rel, 3),
            "RR": reciprocal_rank(ranking, rel),
            "AP": average_precision(ranking, rel),
            "nDCG@5": ndcg_at_k(ranking, relevance[qid], 5),
        })
    frame = pd.DataFrame(rows)
    return frame, frame[["P@3","R@3","RR","AP","nDCG@5"]].mean()

methods = {
    "BM25": lambda q,k: bm25_search(q,k),
    "Dense-like": lambda q,k: latent_search(q,k),
    "Hybrid": lambda q,k: hybrid_search(q,k),
    "Hybrid + reranker": lambda q,k: search_pipeline(q, candidate_k=8, final_k=k),
}

comparison = []
for name, fn in methods.items():
    _, summary = evaluate(fn)
    comparison.append({"method": name, **summary.to_dict()})

comparison_frame = pd.DataFrame(comparison)
comparison_frame

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(comparison_frame["method"], comparison_frame["nDCG@5"])
plt.ylim(0, 1.05)
plt.ylabel("Mean nDCG@5")
plt.title("Retrieval Method Comparison")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()

# 26. Re-Ranking Evaluation

A re-ranker should be judged on whether it improves top-ranked relevance, not merely
whether its classifier accuracy is high.

# 27. Query-Level Analysis

In [ ]:
query_rows = []
for qid, query in queries.items():
    hybrid = hybrid_search(query, 5).doc_id.tolist()
    reranked = search_pipeline(query, 8, 5).doc_id.tolist()
    rel = set(relevance[qid])
    query_rows.append({
        "query_id": qid,
        "query": query,
        "hybrid_RR": reciprocal_rank(hybrid, rel),
        "reranked_RR": reciprocal_rank(reranked, rel),
        "hybrid_nDCG": ndcg_at_k(hybrid, relevance[qid], 5),
        "reranked_nDCG": ndcg_at_k(reranked, relevance[qid], 5),
    })

pd.DataFrame(query_rows)

# 28. Failure Analysis

Common IR failure modes:

- **lexical mismatch** — relevant document uses different terminology;
- **semantic drift** — dense retriever returns related but irrelevant content;
- **rare entity** — important identifier is poorly represented;
- **short query ambiguity** — too little context;
- **domain shift** — index and query vocabularies differ;
- **language mismatch** — query/document representations are incompatible;
- **freshness failure** — index is stale.

# 29. Multilingual Retrieval

Cross-lingual retrieval can use multilingual embeddings, query translation,
language-aware sparse normalization, or cross-lingual re-ranking.

Evaluation should include queries in every target language rather than assuming English
performance transfers automatically.

# 30. Arabic Retrieval

Arabic retrieval must consider:

- morphology;
- clitics;
- orthographic variation;
- optional tashkeel;
- stemming/segmentation policy;
- dialect variation.

# 31. Tashkeel Policy

In [ ]:
ARABIC_DIACRITICS = set("\u064b\u064c\u064d\u064e\u064f\u0650\u0651\u0652")

def strip_tashkeel(text):
    return "".join(ch for ch in text if ch not in ARABIC_DIACRITICS)

arabic_example = "يَعْتَمِدُ الِاسْتِرْجَاعُ النَّصِّيُّ عَلَى التَّشَابُهِ."

pd.Series({
    "original": arabic_example,
    "normalized diagnostic": strip_tashkeel(arabic_example),
})

For a fully vocalized task, preserve the original text for display and primary
evaluation. A normalized form may be indexed as an additional retrieval representation,
but should not silently replace the original.

# 32. Latency

In [ ]:
def latency(search_fn, query, repeats=100):
    values = []
    for _ in range(repeats):
        start = time.perf_counter()
        search_fn(query)
        values.append((time.perf_counter() - start) * 1000)
    return {
        "mean_ms": float(np.mean(values)),
        "p95_ms": float(np.percentile(values, 95)),
    }

pd.DataFrame([
    {"method": "BM25", **latency(lambda q: bm25_search(q,5), queries["q02"])},
    {"method": "Hybrid+reranker", **latency(lambda q: search_pipeline(q,8,5), queries["q02"])},
])

# 33. Caching and Index Freshness

Useful caches include query normalization, embeddings, retrieval results, and reranker
results. They need invalidation policies when indexed content or permissions change.

Production indexes also need incremental additions, deletions, versioning, rollback,
and freshness guarantees.

# 34. Production Architecture

```text
API / UI
  ↓
query service
  ↓
ACL / metadata filter
  ↓
lexical index + vector index
  ↓
fusion
  ↓
reranker
  ↓
result formatter
  ↓
telemetry + relevance evaluation
```

# 35. Monitoring and Access Control

Monitor:

- zero-result rate;
- Recall@k and nDCG on judged sets;
- p50/p95/p99 latency;
- index freshness;
- reranker failures;
- language distribution;
- frequently retrieved hub documents.

Access-control filtering is a correctness requirement: unauthorized documents must not
enter the candidate set.

# 36. Optional Neural Retrieval

The following template is disabled because it requires external models. It shows how
the dense-like SVD and logistic reranker can be replaced by a bi-encoder and
cross-encoder.

In [ ]:
RUN_NEURAL_RETRIEVAL = False

template = '''
from sentence_transformers import SentenceTransformer, CrossEncoder

bi_encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

document_embeddings = bi_encoder.encode(
    document_texts,
    normalize_embeddings=True,
)

query_embedding = bi_encoder.encode(
    [query],
    normalize_embeddings=True,
)

# Retrieve nearest document vectors.

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L6-v2"
)

pairs = [
    [query, document]
    for document in candidate_texts
]

rerank_scores = cross_encoder.predict(pairs)
'''

print(template)

# 37. Reproducibility

In [ ]:
pd.Series({
    "module": "Module 10 • Advanced Applications",
    "lesson": "Lesson 57 • Advanced Information Retrieval",
    "documents": len(docs),
    "queries": len(queries),
    "latent_components": n_components,
    "reranker": "LogisticRegression",
    "seed": SEED,
    "offline_execution": True,
    "python": platform.python_version(),
}, name="Lesson 57 experiment")

# 38. Knowledge Check

1. Why use a multi-stage retrieval pipeline?
2. What does BM25 improve over raw term frequency?
3. What is a bi-encoder?
4. Why are bi-encoders scalable?
5. What is a cross-encoder?
6. Why are cross-encoders used for reranking?
7. What is hybrid retrieval?
8. What does reciprocal-rank fusion combine?
9. What does Recall@k measure?
10. What does MRR emphasize?
11. What does MAP summarize?
12. Why is nDCG useful for graded relevance?
13. What is semantic drift?
14. Why can Arabic normalization change retrieval behavior?
15. What should a production search system monitor?

# 39. Exercises

1. Add 50 documents.
2. Expand the judged query set.
3. Tune BM25 `k1` and `b`.
4. Tune the hybrid lexical weight using development queries.
5. Compare weighted fusion with RRF.
6. Train the reranker on held-out queries.
7. Add query expansion.
8. Add Arabic queries.
9. Compare vocalized and normalized Arabic retrieval.
10. Report Recall@k, MRR, MAP, nDCG, mean latency, and p95 latency.

## Challenge Exercises

1. Replace SVD with SentenceTransformer embeddings.
2. Add a vector index such as FAISS.
3. Add a CrossEncoder reranker.
4. Implement hard-negative mining.
5. Connect the retriever to a RAG pipeline and measure answer quality before and after reranking.

# 40. Summary and Next Lesson

In this lesson:

- BM25-style lexical retrieval was implemented;
- latent-semantic dense-like retrieval was built;
- weighted hybrid fusion and RRF were demonstrated;
- metadata filtering was added;
- a learning-to-rank reranker was trained;
- Precision@k, Recall@k, MRR, MAP, and nDCG were implemented;
- sparse, dense-like, hybrid, and reranked methods were compared;
- Arabic/tashkeel considerations were included;
- latency, caching, freshness, monitoring, and ACL filtering were linked to production search.

## Next Lesson

**Lesson 58: Advanced Text Summarization — Extractive, Abstractive, Long-Document,
and Faithfulness-Aware Summarization**

# References

- Robertson, S. and Zaragoza, H. work on BM25.
- Manning, C. D., Raghavan, P., and Schütze, H. *Introduction to Information Retrieval*.
- Karpukhin, V. et al. *Dense Passage Retrieval for Open-Domain Question Answering*.
- Reimers, N. and Gurevych, I. work on Sentence-BERT.
- Nogueira, R. and Cho, K. work on passage re-ranking.
- Cormack, G. et al. work on Reciprocal Rank Fusion.